# 0.0 Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sklearn.metrics as mt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# 0.1 - Load Datasets

In [2]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [3]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [ ]:
# Instanciar o modelo com parâmetros default (grau 1)
model_def = Pipeline([
    ('features', PolynomialFeatures(degree=1)),
    ('model', LinearRegression())
])

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [ ]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

## Passo 3 — Performance na validação (default)

In [ ]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

## Passo 4 — Ajuste de hiperparâmetros

In [4]:
print("--- Ensaio Isolado: Testando o Impacto dos Graus Polinomiais ---")
melhor_degree = 1
melhor_r2_val = -999

# Lista de graus para testar o ganho de complexidade
list_degrees = [1, 2, 3]

for dg in list_degrees:
    
    # Criamos o algoritmo composto de Regressão Polinomial Pura (Sem freios)
    model_polinomial_puro = Pipeline([
        ('features_polinomiais', PolynomialFeatures(degree=dg, include_bias=False)),
        ('regressor_linear_puro', LinearRegression())
    ])
    
    try:
        # Treinamento direto do Pipeline
        model_polinomial_puro.fit(X_train, y_train.values.ravel())
        
        # Predict
        yhat_train = model_polinomial_puro.predict(X_train)
        yhat_val = model_polinomial_puro.predict(X_val)
        
        # Métricas
        r2_train = mt.r2_score(y_train, yhat_train)
        r2_val = mt.r2_score(y_val, yhat_val)
        rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
        
        print(f"Grau Polinômio: {dg} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f}")
        
        if r2_val > melhor_r2_val:
            melhor_r2_val = r2_val
            melhor_degree = dg
            
    except Exception as e:
        print(f"Erro ao processar o Grau {dg}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO DE GRAUS:")
print(f"Melhor Grau Encontrado: {melhor_degree}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")

--- Ensaio Isolado: Testando o Impacto dos Graus Polinomiais ---
Grau Polinômio: 1 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41
Grau Polinômio: 2 | R² Treino: 0.0942 | R² Val: 0.0665 | RMSE Val: 21.11
Grau Polinômio: 3 | R² Treino: 0.1544 | R² Val: -0.0478 | RMSE Val: 22.37
-> VENCEDOR DO ENSAIO DE GRAUS:
Melhor Grau Encontrado: 2
Maior R² obtido na Validação: 0.0665



## Passo 5 — União treino + validação

In [ ]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

## Passo 6 — Retreino com melhores parâmetros

In [6]:
degree = 2

# Treinando o modelo para ver as outras métricas de avaliação
model_polinomial_puro = Pipeline([
        ('features_polinomiais', PolynomialFeatures(degree=degree, include_bias=False)),
        ('regressor_linear_puro', LinearRegression())
    ])
model_polinomial_puro.fit(X_train, y_train.values.ravel())

# Previsão de treino e validação
yhat_train = model_polinomial_puro.predict(X_train)
yhat_val = model_polinomial_puro.predict(X_val)

#Performance Train
r2_train = mt.r2_score(y_train, yhat_train)
mse_train = mt.mean_squared_error(y_train, yhat_train)
rmse_train = np.sqrt(mse_train)
mae_train = mt.mean_absolute_error(y_train, yhat_train)
mape_train = mt.mean_absolute_percentage_error(y_train, yhat_train)

#Performance Validation
r2_val = mt.r2_score(y_val, yhat_val)
mse_val = mt.mean_squared_error(y_val, yhat_val)
rmse_val = np.sqrt(mse_val)
mae_val = mt.mean_absolute_error(y_val, yhat_val)
mape_val = mt.mean_absolute_percentage_error(y_val, yhat_val)

# Métrica de Treinamento e Teste
print(f"MÉTRICAS TREINAMENTO:")
print(f"R² (Coef. Determinação): {r2_train:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_train:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_train:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_train:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_train * 100:.2f}%")
print("=" * 65)
print(f"MÉTRICAS VALDAÇÃO:")
print(f"R² (Coef. Determinação): {r2_val:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_val:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_val:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_val:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_val * 100:.2f}%")

MÉTRICAS TREINAMENTO:
R² (Coef. Determinação): 0.0942
MSE (Erro Quadrático Médio): 432.99
RMSE (Raiz do Erro Quadrático): 20.81
MAE (Erro Absoluto Médio): 16.46
MAPE (Erro Percentual Médio): 835.05%
MÉTRICAS VALDAÇÃO:
R² (Coef. Determinação): 0.0665
MSE (Erro Quadrático Médio): 445.77
RMSE (Raiz do Erro Quadrático): 21.11
MAE (Erro Absoluto Médio): 16.75
MAPE (Erro Percentual Médio): 854.79%


## Passo 7 — Performance no teste

In [7]:
# ==============================================================================
# JUNTAR AS BASES E EXECUTAR O TESTE FINAL
# ==============================================================================
print("--- Executando o Teste Final ---")

# Juntando treino e validação para dar mais volume de dados ao KNN
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

# Treinando o modelo definitivo com o melhor K encontrado
model_polinomial_puro = Pipeline([
        ('features_polinomiais', PolynomialFeatures(degree=melhor_degree, include_bias=False)),
        ('regressor_linear_puro', LinearRegression())
    ])


model_polinomial_puro.fit(X_train_final, y_train_final.values.ravel())

# Previsão final usando o dataset de teste (que ficou isolado o tempo todo)
yhat_test = model_polinomial_puro.predict(X_test)

#Performance Test
r2_test = mt.r2_score(y_test, yhat_test)
mse_test = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

# Métrica final de produção
print(f"MÉTRICAS NO DATASET DE TESTE:")
print(f"R² (Coef. Determinação): {r2_test:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_test:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_test:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_test:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_test * 100:.2f}%")
print("=" * 65)

--- Executando o Teste Final ---
MÉTRICAS NO DATASET DE TESTE:
R² (Coef. Determinação): 0.0909
MSE (Erro Quadrático Médio): 442.64
RMSE (Raiz do Erro Quadrático): 21.04
MAE (Erro Absoluto Médio): 16.74
MAPE (Erro Percentual Médio): 827.70%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [ ]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   '-', '-', r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, '-', '-', rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  '-', '-', mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%', '-', '-', f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))